# RAG Evaluation with DeepEval

This notebook evaluates the performance of Baseline RAG vs. RAG-Fusion using retrieval metrics (Recall@K, Precision@K, MRR@K) and generation metrics from **DeepEval** (Faithfulness, Answer Relevancy, Contextual Precision, Contextual Recall).

In [1]:
# !pip install deepeval

import pandas as pd
import json
import os
import numpy as np
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

# Disable DeepEval timeouts
os.environ["DEEPEVAL_DISABLE_TIMEOUTS"] = "True"

In [2]:
def load_jsonl(path):
    """Loads a JSONL file into a pandas DataFrame."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return pd.DataFrame(data)

def extract_final_answer(full_text):
    """Trims LLM prompt and only takes the final answer."""
    if not isinstance(full_text, str):
        return ""
    
    # Token penanda di mana asisten mulai menjawab
    split_token = "<|im_start|>assistant\n"
    
    if split_token in full_text:
        # Ambil semua teks setelah token asisten
        return full_text.split(split_token)[-1].strip()
    return full_text.strip()

def calc_retrieval_metrics(row, retrieved_col, k=3):
    """Calculates Precision@K, Recall@K, and MRR@K."""
    retrieved_ids = row[retrieved_col][:k] # Potong hasil sesuai K
    ground_truth_ids = row['ground_truth_ids']
    
    gt_set = set(ground_truth_ids)
    
    # Deteksi mana yang "Hit" (Benar) di urutan top K
    hits = [1 if doc_id in gt_set else 0 for doc_id in retrieved_ids]
    
    # 1. Recall@K = (Jumlah Benar) / (Total Kunci Jawaban)
    recall = sum(hits) / len(gt_set) if len(gt_set) > 0 else 0.0
    
    # 2. Precision@K = (Jumlah Benar) / K
    precision = sum(hits) / k if k > 0 else 0.0
    
    # 3. MRR@K = 1 / (Ranking pertama yang benar)
    mrr = 0.0
    for i, hit in enumerate(hits):
        if hit == 1:
            mrr = 1.0 / (i + 1)
            break
            
    return pd.Series([precision, recall, mrr])

In [3]:
K_VALUE = 3
FILEPATH = f"../knowledge-base/intfloat--multilingual-e5-large_rag_responses_dataset_k3.jsonl"
OUTPUT_FILEPATH = FILEPATH.replace('rag_responses', 'evaluation')

if not os.path.exists(FILEPATH):
    print(f"Error: File {FILEPATH} not found.")
else:
    print(f"1. Loading evaluation data from {FILEPATH}...")
    df = load_jsonl(FILEPATH)
    print(f"Total rows loaded: {len(df)}\n")

1. Loading evaluation data from ../knowledge-base/intfloat--multilingual-e5-large_rag_responses_dataset_k3.jsonl...
Total rows loaded: 50



In [4]:
# Handle nested baseline/fusion columns
if 'baseline' in df.columns and isinstance(df['baseline'].iloc[0], dict):
    df['baseline_answer'] = df['baseline'].apply(lambda x: x.get('answer') if isinstance(x, dict) else None)
    df['baseline_contexts_metadata'] = df['baseline'].apply(lambda x: x.get('metadata') if isinstance(x, dict) else None)
    df['baseline_retrieved_contexts'] = df['baseline'].apply(lambda x: x.get('contexts') if isinstance(x, dict) else None)
    
if 'fusion' in df.columns and isinstance(df['fusion'].iloc[0], dict):
    df['fusion_answer'] = df['fusion'].apply(lambda x: x.get('answer') if isinstance(x, dict) else None)
    df['fusion_contexts_metadata'] = df['fusion'].apply(lambda x: x.get('metadata') if isinstance(x, dict) else None)
    df['fusion_retrieved_contexts'] = df['fusion'].apply(lambda x: x.get('contexts') if isinstance(x, dict) else None)

# Extract Ground Truth IDs from metadata
df['ground_truth_ids'] = df['chunk_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)

# Extract Retrieval IDs from metadata
df['baseline_retrieved_ids'] = df['baseline_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)
df['fusion_retrieved_ids'] = df['fusion_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)

# Clean answers
print("Cleaning answers from prompts...")
df['clean_baseline_answer'] = df['baseline_answer'].apply(extract_final_answer)
df['clean_fusion_answer'] = df['fusion_answer'].apply(extract_final_answer)

Cleaning answers from prompts...


In [5]:
print(f"2. Calculating Retrieval Metrics for K={K_VALUE}...")

df[['baseline_Precision@K', 'baseline_Recall@K', 'baseline_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'baseline_retrieved_ids', k=K_VALUE), axis=1
)
df[['fusion_Precision@K', 'fusion_Recall@K', 'fusion_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'fusion_retrieved_ids', k=K_VALUE), axis=1
)

print("Retrieval Metrics Calculated.")

2. Calculating Retrieval Metrics for K=3...
Retrieval Metrics Calculated.


In [6]:
print("\n3. Preparing data for DeepEval evaluation (LLM-as-a-Judge)...")

def format_contexts(contexts):
    if isinstance(contexts, list):
        return [getattr(c, 'page_content', str(c)) for c in contexts]
    return []

def create_test_cases(dataframe, answer_col, contexts_col):
    test_cases = []
    for i, row in dataframe.iterrows():
        test_case = LLMTestCase(
            input=row["question"],
            actual_output=row[answer_col],
            expected_output=row["ground_truth"],
            retrieval_context=format_contexts(row[contexts_col])
        )
        test_cases.append(test_case)
    return test_cases

test_cases_baseline = create_test_cases(df, "clean_baseline_answer", "baseline_retrieved_contexts")
test_cases_fusion = create_test_cases(df, "clean_fusion_answer", "fusion_retrieved_contexts")

model_name = "gpt-4o-mini"
metrics = [
    FaithfulnessMetric(threshold=0.5, model=model_name),
    AnswerRelevancyMetric(threshold=0.5, model=model_name),
    ContextualPrecisionMetric(threshold=0.5, model=model_name),
    ContextualRecallMetric(threshold=0.5, model=model_name)
]

async_config = AsyncConfig(max_concurrent=3, throttle_value=2)

def extract_results(eval_output):
    test_results = eval_output.test_results
    return [
        {m.name: m.score for m in result.metrics_data}
        for result in test_results
    ]

print("\n> Running DeepEval Evaluation for BASELINE...")
baseline_run_results = evaluate(test_cases_baseline, metrics, async_config=async_config)
baseline_results = extract_results(baseline_run_results)

print("\n> Running DeepEval Evaluation for FUSION...")
fusion_run_results = evaluate(test_cases_fusion, metrics, async_config=async_config)
fusion_results = extract_results(fusion_run_results)


3. Preparing data for DeepEval evaluation (LLM-as-a-Judge)...

> Running DeepEval Evaluation for BASELINE...


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:              Bagaimana hasil monitoring fisik klinis An. MD pada hari kedua berhubungan dengan    │
│  │                         edukasi gizi yang diberikan, terutama mengenai pengetahuan tentang diet seimbang?    │
│  │     Actual Output:      Hasil monitoring fisik klinis An. MD pada hari kedua berhubungan erat dengan         │
│  │                         edukasi gizi yang diberikan, terutama mengenai pengetahuan tentang diet seimbang,    │
│  │                         sebagai berikut:                                                                     │
│  │                                                                                                              │
│  │                         - **Status gizi**: Berdasarkan indikator berat badan per tinggi badan (BB/TB) dan    │
│  │                         indeks massa tubuh (IMT/U), An. MD diklasifikasikan memiliki status gizi kurang.     │
│  │                         Data antropometri menunjukkan berat badan 11,5 kg, tinggi badan 98 cm, dan IMT       │
│  │                         11,3 kg/m², yang merupakan angka di bawah batas normal, menandai risiko tinggi       │
│  │                         terhadap malnutrisi.                                                                 │
│  │                                                                                                              │
│  │                         - **Edukasi gizi yang diberikan**: Edukasi dilaksanakan pada hari kedua setelah      │
│  │                         pasien telah dipersiapkan dalam proses KRS (6 April 2023). Proses ini dimulai        │
│  │                         dengan *pre-test* lisan untuk mengevaluasi pengetahuan keluarga terkait gizi         │
│  │                         sebelum edukasi. Hasil pre-test menunjukkan bahwa keluarga (terutama ibu) awalnya    │
│  │                         tidak memiliki pemahaman yang cukup tentang prinsip diet seimbang, seperti halnya    │
│  │                         dalam pertanyaan tentang tujuan diet, sumber protein, dan makanan tinggi lemak       │
│  │                         baik.                                                                                │
│  │                                                                                                              │
│  │                         - **Peningkatan pengetahuan setelah edukasi**: Setelah edukasi, dilakukan            │
│  │                         *post-test*, dan hasilnya menunjukkan bahwa seluruh pertanyaan berhasil dijawab      │
│  │                         dengan benar. Ini menandakan ad

⚠ WARNING: No hyperparameters logged.
» ]8;id=950341;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 299.12s | token cost: 0.12158190000000003 USD)
» Test Results (50 total tests):
   » Pass Rate: 70.0% | Passed: 35 | Failed: 15

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [7]:
print("\n" + "="*50)
print(f"FINAL EVALUATION RESULTS (Top-K = {K_VALUE})")
print("="*50)

res_b = pd.DataFrame(baseline_results)
res_f = pd.DataFrame(fusion_results)

summary_df = pd.DataFrame({
    "Metrik": [
        f"Recall (Manual)@{K_VALUE}", 
        f"Precision (Manual)@{K_VALUE}", 
        f"MRR (Manual)@{K_VALUE}", 
        "Faithfulness (DeepEval)", 
        "Answer Relevancy (DeepEval)",
        "Contextual Precision (DeepEval)",
        "Contextual Recall (DeepEval)"
    ],
    "RAG Baseline": [
        df['baseline_Recall@K'].mean(),
        df['baseline_Precision@K'].mean(),
        df['baseline_MRR@K'].mean(),
        res_b['Faithfulness'].mean(),
        res_b['Answer Relevancy'].mean(),
        res_b['Contextual Precision'].mean(),
        res_b['Contextual Recall'].mean()
    ],
    "RAG Fusion": [
        df['fusion_Recall@K'].mean(),
        df['fusion_Precision@K'].mean(),
        df['fusion_MRR@K'].mean(),
        res_f['Faithfulness'].mean(),
        res_f['Answer Relevancy'].mean(),
        res_f['Contextual Precision'].mean(),
        res_f['Contextual Recall'].mean()
    ]
})

summary_df


FINAL EVALUATION RESULTS (Top-K = 3)


,Metrik,RAG Baseline,RAG Fusion
0,Recall (Manual)@3,0.200000,0.186667
1,Precision (Manual)@3,0.200000,0.186667
2,MRR (Manual)@3,0.403333,0.290000
3,Faithfulness (DeepEval),0.941775,0.937373
4,Answer Relevancy (DeepEval),0.837232,0.821923
5,Contextual Precision (DeepEval),0.666667,0.741667
6,Contextual Recall (DeepEval),0.790000,0.888333


In [8]:
summary_df.to_csv(OUTPUT_FILEPATH)